In [1]:
from keras.models import Model 
from keras.layers import Input, Convolution2D, MaxPooling2D, Dense, Dropout, Flatten, Dense, Dropout, Flatten, Conv2D, MaxPooling2D
# import np_utils
import numpy as np
import pandas as pd
from keras.callbacks import EarlyStopping
from keras.models import Sequential
from keras.optimizers import Adam
from keras.utils import to_categorical
from sklearn.preprocessing import StandardScaler
import os
import datetime
from sklearn.model_selection import train_test_split

from sklearn.cluster import AgglomerativeClustering
from sklearn.cluster import KMeans
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import SpectralClustering


from sklearn.utils import resample
from sklearn.neighbors import NearestCentroid
from sklearn.metrics import pairwise_distances_argmin_min
from tslearn.metrics import cdist_dtw

In [2]:
poor = pd.read_csv("SimData/bank_reserves_outputs_poor.csv", header=None)
middle = pd.read_csv("SimData/bank_reserves_outputs_middle.csv", header=None)
rich = pd.read_csv("SimData/bank_reserves_outputs_rich.csv", header=None)
sc = StandardScaler()
br = []
for i in np.arange(0, poor.shape[0]):
    sample = pd.concat([poor.iloc[i], middle.iloc[i]], axis=0).T
    sample = pd.concat([sample, rich.iloc[i]], axis=0).T
    sample_std = sc.fit_transform(sample.to_frame())
    br.append(sample_std)

KeyboardInterrupt: 

In [ ]:
ecv_active = pd.read_csv("SimData/epsteinCV_outputs_active.csv", header=None)
ecv_jailed = pd.read_csv("SimData/epsteinCV_outputs_jailed.csv", header=None)
ecv_quiet = pd.read_csv("SimData/epsteinCV_outputs_quiet.csv", header=None)
sc = StandardScaler()
ecv = []
for i in np.arange(0, ecv_active.shape[0]):
    sample = pd.concat([ecv_active.iloc[i], ecv_jailed.iloc[i]], axis=0).T
    sample = pd.concat([sample, ecv_quiet.iloc[i]], axis=0).T
    sample_std = sc.fit_transform(sample.to_frame())
    ecv.append(sample_std)

In [ ]:
def import_ff_data(filename):
    expected_columns=155
    data = []
    with open(filename, 'r') as file:
        for line in file:
            row = line.strip().split(',')
            if len(row) < expected_columns:
                row += [np.nan] * (expected_columns - len(row))
            data.append(row)
    df = pd.DataFrame(data)
    def fill_last_valid(row):
        for i in range(1, len(row)):
            if pd.isna(row[i]):
                row[i] = row[i - 1]  
        return row
    df_filled = df.apply(fill_last_valid, axis=1)
    return df_filled

In [ ]:
ff_onfire = import_ff_data("SimData/forest_fire_outputs_onfire.csv")
print("check 1")
ff_fine = import_ff_data("SimData/forest_fire_outputs_fine.csv")
print("check 2")
ff_burned = import_ff_data("SimData/forest_fire_outputs_burned.csv")
sc = StandardScaler()
ff = []
for i in np.arange(0, ff_onfire.shape[0]):
    sample = pd.concat([ff_onfire.iloc[i], ff_fine.iloc[i]], axis=0).T
    sample = pd.concat([sample, ff_burned.iloc[i]], axis=0).T
    sample_std = sc.fit_transform(sample.to_frame())
    ff.append(sample_std)

In [ ]:
# Place raw features for all three models in a single place so we can iterate over them 

ABMs = ["BR", "ECV", "FF"]
raw_data = []
raw_data.append(br)
raw_data.append(ecv)
raw_data.append(ff)

In [ ]:
RANDOM_STATE = 42

In [ ]:
# Extract PCA embeddings; not re-doing PCA in this notebook
pca_br_embeds = pd.read_csv('extracted_features/bank_reserves_pca_standardized.csv')
pca_br_embeds = pca_br_embeds.drop('Unnamed: 0', axis=1)
pca_br_embeds = pca_br_embeds.to_numpy()

pca_ecv_embeds = pd.read_csv('extracted_features/epstein_pca_standardized.csv')
pca_ecv_embeds = pca_ecv_embeds.drop('Unnamed: 0', axis=1)
pca_ecv_embeds = pca_ecv_embeds.to_numpy()

pca_ff_embeds = pd.read_csv('extracted_features/forestfire_pca_standardized.csv')
pca_ff_embeds = pca_ff_embeds.drop('Unnamed: 0', axis=1)
pca_ff_embeds = pca_ff_embeds.to_numpy()

pca_embeds = []
pca_embeds.append(pca_br_embeds)
pca_embeds.append(pca_ecv_embeds)
pca_embeds.append(pca_ff_embeds)

In [ ]:
# Silouhette score - DTW
from tslearn.clustering import silhouette_score
debug = 1
sil_sample = 20000 # sample 

def sil_score_dtw(X_embed, labels, s): 
    SILH_SAMPLE = s 
    if SILH_SAMPLE is not None and SILH_SAMPLE < X_embed.shape[0]:
        rng = np.random.default_rng(RANDOM_STATE)
        idx = rng.choice(X_embed.shape[0], size=SILH_SAMPLE, replace=False)
        sil = silhouette_score(X_embed[idx], labels[idx], metric="dtw")
    else:
        sil = silhouette_score(X_embed, labels, metric="dtw")

    unique, counts = np.unique(labels, return_counts=True)
    cluster_sizes = dict(zip(unique.tolist(), counts.tolist()))
    if debug:
        print(f"Silhouette Score: {sil:.4f}")
      #  print("Cluster sizes:", cluster_sizes)
    return sil

In [ ]:
# Silouhette score 

from sklearn.metrics import silhouette_score
debug = 1
sil_sample = 20000 # sample 

def sil_score(X_embed, labels, s): 
    SILH_SAMPLE = s 
    if SILH_SAMPLE is not None and SILH_SAMPLE < X_embed.shape[0]:
        rng = np.random.default_rng(RANDOM_STATE)
        idx = rng.choice(X_embed.shape[0], size=SILH_SAMPLE, replace=False)
        sil = silhouette_score(X_embed[idx], labels[idx], metric="euclidean")
    else:
        sil = silhouette_score(X_embed, labels, metric="euclidean")

    unique, counts = np.unique(labels, return_counts=True)
    cluster_sizes = dict(zip(unique.tolist(), counts.tolist()))
    if debug:
        print(f"Silhouette Score: {sil:.4f}")
      #  print("Cluster sizes:", cluster_sizes)
    return sil

In [ ]:
# Get silhouette scores for raw features
for i, abm in enumerate(ABMs): 
    for k in range(3,11):
        flat_list = [sample.flatten() for sample in raw_data[i]]
        raw_features = np.asarray(flat_list)
        raw_labels = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10).fit(raw_features).labels_
        unique_labels = np.unique(raw_labels)
        if (len(unique_labels) > 1): 
            score = sil_score(pca_embeds[i], raw_labels, sil_sample)
        else: 
            print(f"Error: Too few labels generated. Skipping ...")
        print(f"[PCA,{abm},{k}] \t Silhouette Score: {score:.4f}")  
        with open("raw_sil_scores.csv", "a") as f:
            f.write(f"{abm},{k},{score:.4f}\n")

In [ ]:
# Get silhouette scores for pca features 
for i, abm in enumerate(ABMs): 
    for k in range(3,11):
        pca_labels = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10).fit(pca_embeds[i]).labels_
        unique_labels = np.unique(pca_labels)
        if (len(unique_labels) > 1): 
            score = sil_score(pca_embeds[i], pca_labels, sil_sample)
        else: 
            print(f"Error: Too few labels generated. Skipping ...")
        print(f"[PCA,{abm},{k}] \t Silhouette Score: {score:.4f}")
        with open("pca_sil_scores.csv", "a") as f:
            f.write(f"{abm},{k},{score:.4f}\n")

In [ ]:
## DAE

import tensorflow as tf

def DAE_reduction(df, bottleneck, ep, batch):
    df_flat = [sample.flatten() for sample in df]
    df_flat = np.asarray(df_flat)
    train, test = train_test_split(df_flat, test_size=0.20, random_state=42)
    input = Input(shape=(df_flat.shape[1],))

    encoded = Dense(30, activation='relu')(input)
    encoded = Dense(20, activation='relu')(encoded)
    encoded = Dense(10, activation='relu')(encoded)
    encoded = Dense(bottleneck, activation='linear', name='bottleneck')(encoded)

    decoded = Dense(bottleneck, activation='relu')(encoded)
    decoded = Dense(20, activation='relu')(decoded)
    decoded = Dense(30, activation='relu')(decoded)
    output = Dense(df_flat.shape[1], activation=None)(decoded)

    autoencoder = Model(input, output)
#   autoencoder.summary()
#   autoencoder.compile(optimizer=Adam(learning_rate=0.001), loss='mse')
    huber = tf.keras.losses.Huber()
    autoencoder.compile(optimizer=Adam(learning_rate=0.001), loss=huber)
    autoencoder.fit(train, train,
     epochs=ep,
     batch_size=batch,
     shuffle=True,
     validation_data=(test, test), verbose = 0)
    encoder = Model(inputs=autoencoder.input, outputs=autoencoder.get_layer('bottleneck').output)
    encoded_ts = encoder.predict(df_flat)
    return encoded_ts, encoder 

In [ ]:
opt_clusters = []
opt_clusters.append(7)
opt_clusters.append(8)
opt_clusters.append(4)

In [ ]:
# Hyperparameter tuning; 
# hardcoded for DAE with linear bottleneck layer, 2048 batch size and Huber loss; 
# may want to tune them as well

def hypertune(target, k, clustering, abm): 

    BOTTLENECK_MIN = 5 
    EPOCH_MIN = 3
    
    bottleneck = 5     #  number of desired embeddings; tune in the range of [BOTTLENECK_MIN..bottleneck]
    epochs = 3         #  tune in range of [EPOCH_MIN..epochs] 
    batch = 2048       #  fixed from prior trials 
    evals = 10           #  number of repeated trainings to reduce noise in model 
  
    sil_sample = 20000 # silohouette sample 

    max_sil_score = 0         # max silohouette score 
    best_embed = []           # best emebeddings 
    best_model = []           # model that produced the best embeddings 
    
    for b in np.arange(BOTTLENECK_MIN, bottleneck + 1):   # bottleneck 
        for e in np.arange(EPOCH_MIN, epochs + 1):        # epoch
            for i in np.arange(1, evals + 1):             # evals 
                
                print(f"Evaluation: embeddings {b}, epoch {e}, eval # {i}:")
                # get embeddings 
                dae_embeds, dae_model = DAE_reduction(raw_data[target], bottleneck, epochs, batch)

                # clustering 
                if (clustering == 'agglom'):
                 #   dae_labels = agglomerative(dae_embeds, k, 'euclidean', 'knn')
                    dae_labels = agglom_dtw(dae_embeds, k, 'dtw')
                else: 
                    dae_labels = KMeans(n_clusters=k, random_state=42, n_init=10).fit(dae_embeds).labels_
                
                unique_labels = np.unique(dae_labels)
                if (len(unique_labels) > 1): 
                    # silhouette score 
                    score = sil_score(dae_embeds, dae_labels, sil_sample)
                    # save the best embeddings and model   
                    if (score > max_sil_score): 
                        max_sil_score = score
                        best_embed = dae_embeds
                        best_model.append(dae_model) # assuming we are hypertuning the three models in order  
                        best_params = str(b) + '_' + str(e) + '_' + str(batch) + '_linear'
                    # log result 
                    outfile = 'dae_sil_scores_all_dtw_' + clustering + '.csv'
                    with open(outfile, "a") as f:
                        f.write(f"{abm},{k},{b},{e},{i},{score:.4f}\n")
                else: 
                    print(f"Error: Too few labels generated. Skipping ...")
 
    print(f"Best Silhouette: {max_sil_score:.4f}")
    print(f"Best parameters: {best_params}")
    filename = 'extracted_features/' + str(target) + '_DAE_' + best_params + '_' + clustering + '.csv'
    np.savetxt(filename, best_embed, delimiter=",")
    with open("dae_sil_scores.csv", "a") as f:
        f.write(f"{abm},{k},{max_sil_score:.4f}\n")

In [ ]:
# Hypertuning driver 
for i, abm in enumerate(ABMs): 
    for k in range(3,4):
        hypertune(i, k, 'agglom', abm)

In [ ]:
# validate 
ff_dae = pd.read_csv(filename, header=None)
X_embed = ff_dae.to_numpy()
ff_dae_kmeans = KMeans(n_clusters=4, random_state=0, n_init=10).fit(ff_dae).labels_
labels = ff_dae_kmeans
sil_score(X_embed, labels, 20000)

In [ ]:
from sklearn.metrics import pairwise_distances_argmin_min

def agglomerative(embeds, k, distance, assign_method):

    sample_size = 5000 
    sampled_embeds = resample(embeds, n_samples=sample_size, random_state=42)
    
    hierarchical_cluster = AgglomerativeClustering(n_clusters=k, metric=distance, linkage='ward')
    labels_sub = hierarchical_cluster.fit_predict(sampled_embeds)
    
    if (assign_method == 'knn'): 
        neigh = KNeighborsClassifier(n_neighbors=13)
        neigh.fit(sampled_embeds, labels_sub)
        labels = neigh.predict(embeds) 
    else:      
        centroids_model = NearestCentroid()
        centroids_model.fit(sampled_embeds, labels_sub)
        cluster_centroids = centroids_model.centroids_
        labels, _  = pairwise_distances_argmin_min(embeds, cluster_centroids)

    return labels

In [ ]:
def agglom_dtw(embeds, k, distance):

    embeds_3D = np.expand_dims(embeds, axis=-1)
    sample_size = 1000  
    sampled_embeds = resample(embeds_3D, n_samples=sample_size, random_state=42)

    dtw_distance_matrix = cdist_dtw(sampled_embeds)

    hc = AgglomerativeClustering(n_clusters=k, metric='precomputed', linkage='average')
    sampled_labels = hc.fit_predict(dtw_distance_matrix)

 
    cluster_centroids = []
    for i in range(k):
        cluster_indices = np.where(sampled_labels == i)[0]
        cluster_series = sampled_embeds[cluster_indices]
        dist_matrix_cluster = cdist_dtw(cluster_series)
        medoid_idx_in_cluster = np.argmin(dist_matrix_cluster.sum(axis=1))
        cluster_centroids.append(cluster_series[medoid_idx_in_cluster])
        
    cluster_centroids = np.array(cluster_centroids)  
    
    from tslearn.clustering import TimeSeriesKMeans

    ts_kmeans = TimeSeriesKMeans(
        n_clusters=k,
        metric="dtw",
        max_iter=0,
        random_state=42
    )
    ts_kmeans.cluster_centers_ = cluster_centroids

    labels = ts_kmeans.predict(embeds_3D)
    
    return labels


In [ ]:
ecv_dae = DAE_reduction(ecv, 5, 2, 2048)

In [ ]:
agglomerative(dae_ecv, 4, 'euclidean', 'ward')

In [ ]:
import numpy as np
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from sklearn.utils import resample
import matplotlib.pyplot as plt

# Generate a small dataset for demonstration
np.random.seed(42)
embeddings_small = np.random.rand(50, 5)

# Perform hierarchical clustering using SciPy's `linkage` function.
# 'ward' linkage minimizes the variance of the clusters being merged.
linkage_matrix = linkage(embeddings_small, method='ward', metric='euclidean')

# Plot the dendrogram to visualize the hierarchy
plt.figure(figsize=(10, 7))
plt.title("Dendrogram for a small sample")
dendrogram(linkage_matrix)
plt.show()

# Use fcluster to extract flat clusters from the linkage matrix
# `t` is the threshold to apply when forming flat clusters.
# We can use the dendrogram to visually determine an appropriate threshold.
# Here, we will cut the tree to produce 3 clusters.
num_clusters = 3
labels_small = fcluster(linkage_matrix, num_clusters, criterion='maxclust')

print(f"Labels for the small sample: {labels_small}")


In [202]:
ff_dae = DAE_reduction(ff, 8)

3125/3125 ━━━━━━━━━━━━━━━━━━━━ 1s 465us/step


In [189]:
from sklearn.cluster import KMeans
ff_pca_kmeans = KMeans(n_clusters=4, random_state=0, n_init=10).fit(ff_pca).labels_
ff_dae_kmeans = KMeans(n_clusters=4, random_state=0, n_init=10).fit(ff_dae).labels_
ff_dcae_kmeans = KMeans(n_clusters=4, random_state=0, n_init=10).fit(ff_dcae).labels_

In [74]:
ff_pca = pd.read_csv('extracted_features/forestfire_pca_standardized.csv')
ff_pca = ff_pca.drop('Unnamed: 0', axis=1)
#ff_dae = pd.read_csv('extracted_features/forestfire_dae.csv', header=None)
#ff_dcae = pd.read_csv('extracted_features/forestfire_dcae.csv', header=None)

In [32]:
np.savetxt("extracted_features/bank_reserves_DAE.csv", br_dae, delimiter=",")
np.savetxt("extracted_features/epstein_DAE.csv", ecv_dae, delimiter=",")
np.savetxt("extracted_features/ff_DAE.csv", ff_dae, delimiter=",")